# Study 871 — The Rank Effect — the teardown

The per-leg splits, the Newey-West spread *t*, the level-controlled residual spread, the pooled Welch book test, the 1,000-permutation placebo, the two-era robustness cut, the costed timer, and the 20-seed synthetic control.

In [1]:
R = {'start': '2010-01-04', 'end': '2026-06-30', 'n_names': 50, 'n_days': 4104, 'spread_bps': -1.65, 't_nw': -1.86, 't_1s': -1.88, 'mid_bps': 6.71, 'ext_bps': 8.36, 'welch_t': -0.64, 'gross_sharpe': -0.46, 'lvl_mid_pct': 2.47, 'lvl_ext_pct': 3.87, 'lc_spread_bps': 0.11, 'lc_t_nw': 0.18, 'lc_t_1s': 0.17, 'placebo_obs': -1.65, 'placebo_mean': 0.036, 'placebo_sd': 0.789, 'placebo_p': 0.981, 'placebo_sigma_left': 2.14, 'placebo_draws': 1000, 'era_early_bps': -0.58, 'era_early_t': -0.5, 'era_early_n': 1970, 'era_late_bps': -2.63, 'era_late_t': -1.99, 'era_late_n': 2134, 'timer_1_gross': -1.65, 'timer_1_cost': 2.14, 'timer_1_net': -3.79, 'timer_1_t': -4.31, 'timer_5_gross': -1.65, 'timer_5_cost': 10.14, 'timer_5_net': -11.79, 'timer_5_t': -13.42, 'null_mean_t': 0.24, 'null_sd_t': 0.96, 'null_fire': 1, 'planted_raw_t': 4.14, 'planted_lc_t': 2.33}

## The headline — long-middle / short-extremes spread

Daily equal-weight middle-40% minus both-20%-tails rank-extremity spread.

In [2]:
print(f"spread        : {R['spread_bps']:+.2f} bps/day  NW(10) t = {R['t_nw']:+.2f}  "
      f"one-sample t = {R['t_1s']:+.2f}")
print(f"books         : middle {R['mid_bps']:+.2f} vs extremes {R['ext_bps']:+.2f} bps "
      f"(Welch t = {R['welch_t']:+.2f})")
print(f"level neutral : middle trail {R['lvl_mid_pct']:+.2f}%  vs extremes {R['lvl_ext_pct']:+.2f}%")
print(f"gross Sharpe  : {R['gross_sharpe']:+.2f} (before cost)")

spread        : -1.65 bps/day  NW(10) t = -1.86  one-sample t = -1.88
books         : middle +6.71 vs extremes +8.36 bps (Welch t = -0.64)
level neutral : middle trail +2.47%  vs extremes +3.87%
gross Sharpe  : -0.46 (before cost)


## Controlling for the raw return level — the effect vanishes

Residualise each day's forward return on a quadratic in the standardised trailing-return level (remove *any* smooth momentum/reversal curve), then re-measure the middle-minus-extremes spread. What survives is the pure rank-*position* effect.

In [3]:
print(f"level-controlled spread: {R['lc_spread_bps']:+.2f} bps/day  NW(10) t = {R['lc_t_nw']:+.2f}  "
      f"(one-sample t = {R['lc_t_1s']:+.2f})  -> a flat zero")

level-controlled spread: +0.11 bps/day  NW(10) t = +0.18  (one-sample t = +0.17)  -> a flat zero


## Placebo — column-permute the forward returns (1,000 permutations)

In [4]:
print(f"observed {R['placebo_obs']:+.2f} bps vs placebo mean {R['placebo_mean']:+.3f} "
      f"(sd {R['placebo_sd']:.3f}) -> right-tail p = {R['placebo_p']:.5f} "
      f"(~{R['placebo_sigma_left']:.2f}sigma into the WRONG (left) tail)")

observed -1.65 bps vs placebo mean +0.036 (sd 0.789) -> right-tail p = 0.98100 (~2.14sigma into the WRONG (left) tail)


## Robustness — two eras (split 2018-01-01)

In [5]:
print(f"2010-2017 (n={R['era_early_n']}): {R['era_early_bps']:+.2f} bps  NW t = {R['era_early_t']:+.2f}")
print(f"2018-2026 (n={R['era_late_n']}): {R['era_late_bps']:+.2f} bps  NW t = {R['era_late_t']:+.2f}")

2010-2017 (n=1970): -0.58 bps  NW t = -0.50
2018-2026 (n=2134): -2.63 bps  NW t = -1.99


## The timer — can you get paid for it?

2 sides × one-way cost × NAV per day on the long-short book; short pays 50 bps/yr borrow.

In [6]:
for tag,g,c,n,t in [('1 bp',R['timer_1_gross'],R['timer_1_cost'],R['timer_1_net'],R['timer_1_t']),
                    ('5 bps',R['timer_5_gross'],R['timer_5_cost'],R['timer_5_net'],R['timer_5_t'])]:
    print(f"{tag:>5} one-way: gross {g:+.2f} -> net {n:+.2f} bps/day (cost {c:.2f}/day, t={t:+.2f})")

 1 bp one-way: gross -1.65 -> net -3.79 bps/day (cost 2.14/day, t=-4.31)
5 bps one-way: gross -1.65 -> net -11.79 bps/day (cost 10.14/day, t=-13.42)


## Synthetic positive control — the machinery is unbiased

Live: the detector must NOT fire on the null and must recover a planted relation.

In [7]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from rank_effect import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=871+s, n_assets=40, n_days=1200))['lc_t_nw'] for s in range(8)])
print(f"null (edge=0), 8 seeds: level-ctrl NW t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/8")
planted = st.synthetic_detect(data.synthetic_panel(edge=0.0016, seed=871, n_assets=40, n_days=1500))
print(f"planted (edge=0.0016): raw NW t = {planted['t_nw']:+.2f}, level-ctrl NW t = {planted['lc_t_nw']:+.2f}")

null (edge=0), 8 seeds: level-ctrl NW t mean +0.53 (sd 1.07), |t|>=2 in 1/8


planted (edge=0.0016): raw NW t = +4.14, level-ctrl NW t = +2.33


## Verdict

- **Signal — None.** Hartzmark's rank effect leaves **no** cross-sectional return footprint on 50 liquid US mega-caps. The specified long-middle / short-extremes spread is **-1.65 bps/day** (NW *t* = **-1.86**) — *wrong-signed* and insignificant — and once you **control for the raw return level** it collapses to **+0.11 bps/day** (*t* = +0.18), a flat zero. Not robust across eras (*t* = -0.50 / -1.99); the observed value sits ~2.14σ into the *wrong* tail of a 1,000-permutation placebo. The 20-seed synthetic control recovers a *planted* relation cleanly (level-ctrl *t* = +2.33, fires on 1/20 nulls), so the absence is real, not machinery.
- **Tradability — Mirage.** The specified book loses money gross and net (**-3.79 bps/day** at 1 bp one-way, -11.79 at 5 bps); even the data-mined sign-flip is eaten by the 2.14 bps/day round-trip friction at 1 bp.